# Supervised Learning for Review Classification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

# Custom modules
import sys
sys.path.append('../')
from utils.preprocessing import preprocess_text

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
# Load data
df = pd.read_csv('../data/processed/cleaned_reviews.csv')
print(f"Dataset shape: {df.shape}")
display(df.head())

# Prepare data for classification
X = df['cleaned_review']
y_rating = df['rating']  # For regression or classification
y_sentiment = df['rating'].apply(lambda x: 'positive' if x >= 4 else 'negative')  # Binary sentiment
y_category = df['category']

# Encode labels
le_sentiment = LabelEncoder()
y_sentiment_encoded = le_sentiment.fit_transform(y_sentiment)

le_category = LabelEncoder()
y_category_encoded = le_category.fit_transform(y_category)

print(f"Sentiment classes: {le_sentiment.classes_}")
print(f"Category classes: {le_category.classes_}")

# Split data
X_train, X_test, y_train_sent, y_test_sent = train_test_split(X, y_sentiment_encoded, test_size=0.2, random_state=42)
X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(X, y_category_encoded, test_size=0.2, random_state=42)

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Model 1: Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train_sent)
nb_pred = nb_model.predict(X_test_tfidf)

print("Naive Bayes Results:")
print(f"Accuracy: {accuracy_score(y_test_sent, nb_pred):.4f}")
print(classification_report(y_test_sent, nb_pred, target_names=le_sentiment.classes_))

# Model 2: Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train_sent)
lr_pred = lr_model.predict(X_test_tfidf)

print("\nLogistic Regression Results:")
print(f"Accuracy: {accuracy_score(y_test_sent, lr_pred):.4f}")
print(classification_report(y_test_sent, lr_pred, target_names=le_sentiment.classes_))

# Save models
with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('../models/nb_model.pkl', 'wb') as f:
    pickle.dump(nb_model, f)
with open('../models/lr_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)

print("\nModels saved.")

In [ ]:
# Neural Network Model
# Tokenize for NN
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad = pad_sequences(X_test_seq, maxlen=100)

# Build model
model = Sequential([
    Embedding(5000, 128, input_length=100),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train
history = model.fit(X_train_pad, y_train_sent, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate
loss, accuracy = model.evaluate(X_test_pad, y_test_sent)
print(f"Neural Network Accuracy: {accuracy:.4f}")

# Save model
model.save('../models/nn_model.h5')
with open('../models/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Neural network model saved.")

In [ ]:
# Model 3: SVM
svm_model = SVC(kernel='linear', C=1.0)
svm_model.fit(X_train_tfidf, y_train_sent)
svm_pred = svm_model.predict(X_test_tfidf)

print("\nSVM Results:")
print(f"Accuracy: {accuracy_score(y_test_sent, svm_pred):.4f}")
print(classification_report(y_test_sent, svm_pred, target_names=le_sentiment.classes_))

# Save SVM model
with open('../models/svm_model.pkl', 'wb') as f:
    pickle.dump(svm_model, f)

print("SVM model saved.")

In [ ]:
# Model 4: LSTM
lstm_model = Sequential([
    Embedding(5000, 128, input_length=100),
    LSTM(64, return_sequences=False),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()

# Train LSTM
lstm_history = lstm_model.fit(X_train_pad, y_train_sent, epochs=5, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate
lstm_loss, lstm_accuracy = lstm_model.evaluate(X_test_pad, y_test_sent)
print(f"LSTM Accuracy: {lstm_accuracy:.4f}")

# Save LSTM model
lstm_model.save('../models/lstm_model.h5')
print("LSTM model saved.")

In [ ]:
# Category Classification
# TF-IDF for categories
X_train_cat_tfidf = tfidf.transform(X_train_cat)
X_test_cat_tfidf = tfidf.transform(X_test_cat)

# Model for categories: Logistic Regression
cat_model = LogisticRegression(max_iter=1000, multi_class='ovr')
cat_model.fit(X_train_cat_tfidf, y_train_cat)
cat_pred = cat_model.predict(X_test_cat_tfidf)

print("\nCategory Classification Results (Logistic Regression):")
print(f"Accuracy: {accuracy_score(y_test_cat, cat_pred):.4f}")
print(classification_report(y_test_cat, cat_pred, target_names=le_category.classes_))

# Save category model
with open('../models/cat_model.pkl', 'wb') as f:
    pickle.dump(cat_model, f)
with open('../models/le_category.pkl', 'wb') as f:
    pickle.dump(le_category, f)

print("Category model saved.")

In [ ]:
# Model Comparison
models = ['Naive Bayes', 'Logistic Regression', 'SVM', 'Neural Network', 'LSTM']
accuracies = [
    accuracy_score(y_test_sent, nb_pred),
    accuracy_score(y_test_sent, lr_pred),
    accuracy_score(y_test_sent, svm_pred),
    accuracy,
    lstm_accuracy
]

plt.figure(figsize=(10, 6))
sns.barplot(x=models, y=accuracies)
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f'{v:.4f}', ha='center')
plt.show()

# Error Analysis
print("\nError Analysis for Logistic Regression:")
errors = X_test[lr_pred != y_test_sent]
error_preds = lr_pred[lr_pred != y_test_sent]
error_true = y_test_sent[lr_pred != y_test_sent]

for i in range(min(5, len(errors))):
    print(f"Text: {errors.iloc[i]}")
    print(f"Predicted: {le_sentiment.inverse_transform([error_preds[i]])[0]}, True: {le_sentiment.inverse_transform([error_true[i]])[0]}")
    print("---")